# Annotate OMERO images with micro-SAM

Pick images from an OMERO dataset, project, plate or screen, annotate them in napari with micro-SAM, and upload the masks back to OMERO as ROIs. A tracking table on the container records which images are done, so you can stop and resume later.

Run the cells from top to bottom.

## 1. Import

In [ ]:
import omero_annotate_ai
from omero_annotate_ai import (
    create_omero_connection_widget,
    create_pipeline,
    create_workflow_widget,
)

print(f"omero-annotate-ai {omero_annotate_ai.__version__}")

## 2. Connect to OMERO

Fill in the server and your credentials, then click **Connect**.

In [ ]:
conn_widget = create_omero_connection_widget()
conn_widget.display()

In [ ]:
conn = conn_widget.get_connection()
if conn is None:
    raise ConnectionError(
        "Not connected yet: use the widget above, then rerun this cell."
    )
print("Connected to OMERO.")

## 3. Configure the annotation run

Work through the tabs of the widget:

1. **Working directory**: where images, masks and the config are saved locally.
2. **Container**: the dataset, project, plate or screen to annotate.
3. **Tables**: click **Scan for Tables**. To continue an earlier run, select its table and click **Continue Existing**. To start fresh, click **Name New Table**.
4. **Parameters**: channel, model, number of images, patches, 3D. Click **Update Configuration** when done.
5. **Save** (optional): write the configuration to a YAML file.

In [ ]:
workflow_widget = create_workflow_widget(connection=conn)
workflow_widget.display()

## 4. Check the configuration

In [ ]:
config = workflow_widget.get_config()

print(f"Container:   {config.omero.container_type} {config.omero.container_id}")
print(f"Table:       {config.name}")
print(f"Resume:      {config.workflow.resume_from_table}")
print(f"Read-only:   {config.workflow.read_only_mode}")
print(f"Channel:     {config.spatial_coverage.channels}")
print(f"Model:       {config.ai_model.pretrained_from}")
print(f"Output:      {config.output.output_directory}")

### Optional: settings the widget does not show

Uncomment what you need before running the pipeline.

In [ ]:
# Only annotate wells whose key-value pairs match (plates and screens)
# config.omero.well_filters = {"cellline": ["U2OS"], "treatment": ["Control"]}
# config.omero.well_filter_mode = "include"  # or "exclude"

# Show extra channels in napari next to the annotated one
# config.spatial_coverage.context_channels = [0, 2]
# config.spatial_coverage.context_channel_names = ["DAPI", "RFP"]
# config.spatial_coverage.context_channel_colormaps = ["blue", "red"]

## 5. Annotate

Images are loaded and napari opens. Annotate each image, then close napari: the masks are uploaded to OMERO as ROIs and the tracking table is updated. The configuration is saved as a YAML file in the output folder; reload it later with `omero_annotate_ai.load_config(path)`.

In [ ]:
pipeline = create_pipeline(config, conn)
table_id, config = pipeline.run_full_micro_sam_workflow()

print(f"Tracking table ID: {table_id}")
print(f"Annotated: {len(config.get_processed())} of {len(config.annotations)}")
if config.workflow.read_only_mode:
    print(f"Masks saved locally in {config.output.output_directory}")

## 6. Close the connection

In [ ]:
conn.close()
print("OMERO connection closed.")